# Assistente RAG do Código de Defesa do Consumidor (Chat com CDC)

Este notebook implementa um pipeline de **Retrieval-Augmented Generation (RAG)** especializado no **Código de Proteção e Defesa do Consumidor (CDC)**.

### Objetivo:
Permitir que o consumidor consulte situacoes reais de compra (lojas, compras online, produtos com defeito, cobrancas indevidas, prazos de garantia) e receba uma orientacao confiavel, fundamentada no CDC e com **citacao expressa da pagina do PDF e do artigo legal**, sem risco de alucinacao.

### Etapas:
1. **Instalacao:** Bibliotecas `pymupdf` (`fitz`), `PyPDF2`, `google-genai`, `rank-bm25`, `numpy` e `pydantic`.
2. **Autenticacao:** Conexao com a Google Gemini API via chave gratuita do Google AI Studio.
3. **Extracao de Alta Fidelidade:** Leitura pagina a pagina do PDF do CDC utilizando `PyMuPDF` (`fitz`) com fallback para `PyPDF2`.
4. **Chunking Juridico Estruturado:** Segmentacao inteligente por **Artigos (`Art. X`)**, preservando *caput*, paragrafos (`§`), incisos e metadados hierarquicos (`titulo`, `capitulo`, `secao`, `artigo`, `page`, `pages`).
5. **Busca Hibrida Real (Embeddings + BM25 + RRF):** Combinacao de busca semantica vetorial (`gemini-embedding-001`) com busca lexical probabilistica (`BM25Okapi`) via fusao de ranking (*Reciprocal Rank Fusion*).
6. **Saida Estruturada com Pydantic (Gemini 3.5 Flash-Lite):** Geracao validada por JSON Schema (`RespostaJuridicaCDC`) separando fundamentacao, orientacao clara, artigos citados, paginas exatas e trecho literal da lei.

## 1. Instalacao das Dependencias

In [ ]:
!pip install -q -U google-genai pymupdf PyPDF2 rank-bm25 numpy pydantic

## 2. Autenticacao Nativa da API do Gemini

A chave e obtida de forma nativa e automatica via Secrets do Colab (`userdata.get('GEMINI_API_KEY')`), variaveis de ambiente (`os.environ`) ou arquivo `.env` local, sem necessidade de digitacao interativa.

In [ ]:
import os
from pathlib import Path
from google import genai

PROJECT_ROOT = next(
    (directory for directory in (Path.cwd(), *Path.cwd().parents)
     if (directory / "backend").is_dir() and (directory / "data").is_dir()),
    Path.cwd(),
)
env_path = PROJECT_ROOT / ".env"
api_key = None
try:
    from google.colab import userdata
    api_key = userdata.get('GEMINI_API_KEY')
except (ImportError, Exception):
    pass

if not api_key:
    api_key = os.environ.get('GEMINI_API_KEY')

if not api_key and env_path.is_file():
    try:
        with env_path.open('r', encoding='utf-8') as file:
            for line in file:
                line = line.strip()
                if line.startswith('GEMINI_API_KEY='):
                    api_key = line.split('=', 1)[1].strip(" \"'")
                    if api_key:
                        break
    except OSError:
        pass

client = genai.Client(api_key=api_key) if api_key else genai.Client()
print("Cliente Gemini configurado.")

## 3. Base Documental Nativa do CDC

O documento oficial do Código de Defesa do Consumidor (ed. PROCON-SP 2025) já está integrado nativamente ao repositório.

In [ ]:
from pathlib import Path

pdf_candidates = [
    PROJECT_ROOT / "data" / "documents" / "codigo_defesa_consumidor.pdf",
    Path.cwd() / "codigo_defesa_consumidor.pdf",
    Path.cwd() / "CDC_2025.pdf",
]
pdf_path = next((path for path in pdf_candidates if path.is_file()), None)
if pdf_path is None:
    raise FileNotFoundError("PDF do CDC nao encontrado em data/documents ou na pasta atual do Colab.")
pdf_filename = str(pdf_path)
print(f"Documento oficial do CDC carregado: {pdf_path.name}")

## 4. Extracao de Alta Fidelidade e Chunking Juridico Estruturado (Artigos, Paragrafos e Incisos)

Diferente de um divisor de texto comum que corta frases no meio ou separa o *caput* do artigo de seus paragrafos (`§ 1º`, `Paragrafo unico`), este parser juridico:
- Extrai o texto pagina a pagina via **PyMuPDF (`fitz`)** (com fallback para `PyPDF2`), corrigindo hifenizacoes de quebra de linha.
- Rastreia a hierarquia normativa (**TÍTULO**, **CAPÍTULO** e **SEÇÃO**).
- Agrupa cada **Artigo (`Art. X`)** integralmente junto com seus incisos, alineas e paragrafos, mesmo quando o artigo atravessa a virada de uma pagina para outra, preservando a pagina exata de inicio (`page`) e a lista de paginas abrangidas (`pages`).

In [ ]:
import re
import PyPDF2

try:
    import fitz  # PyMuPDF
    USAR_PYMUPDF = True
except ImportError:
    USAR_PYMUPDF = False

REGEX_TITULO = re.compile(r'^(T[IÍ]TULO\s+[IVXLCDM]+(?:\s*[-–—].*)?)$', re.IGNORECASE)
REGEX_CAPITULO = re.compile(r'^(CAP[IÍ]TULO\s+[IVXLCDM]+(?:\s*[-–—].*)?)$', re.IGNORECASE)
REGEX_SECAO = re.compile(r'^(SE[CÇ][AÃ]O\s+[IVXLCDM]+(?:\s*[-–—].*)?)$', re.IGNORECASE)
REGEX_INICIO_ARTIGO = re.compile(
    r'(?:^|\n)\s*(Art\.?\s*\d+(?:-[A-Z])?\s*[º°]?\s*[.\-–]?)',
    re.IGNORECASE
)

def _limpar_texto_pagina(texto: str) -> str:
    if not texto:
        return ""
    # Remove hifenizacao de palavras quebradas no fim da linha
    texto = re.sub(r'(\w)-\s*\n\s*(\w)', r'\1\2', texto)
    # Normaliza espacos horizontais multiplos preservando quebras de linha
    texto = re.sub(r'[ \t]+', ' ', texto)
    return texto.strip()

def _extrair_paginas_pdf(caminho_pdf: str) -> list[tuple[int, str]]:
    paginas = []
    if USAR_PYMUPDF:
        with fitz.open(caminho_pdf) as doc:
            for num_pagina, pagina in enumerate(doc, start=1):
                texto = _limpar_texto_pagina(pagina.get_text("text") or "")
                if texto:
                    paginas.append((num_pagina, texto))
    else:
        reader = PyPDF2.PdfReader(caminho_pdf)
        for num_pagina, pagina in enumerate(reader.pages, start=1):
            texto = _limpar_texto_pagina(pagina.extract_text() or "")
            if texto:
                paginas.append((num_pagina, texto))
    return paginas

def _normalizar_rotulo_artigo(cabecalho_art: str) -> str:
    m = re.search(r'(\d+(?:-[A-Z])?)', cabecalho_art, re.IGNORECASE)
    if not m:
        return cabecalho_art.strip()
    num = m.group(1).upper()
    return f"Art. {num}"

def extrair_chunks_cdc(caminho_pdf: str, max_chars_chunk: int = 1800) -> list[dict]:
    paginas = _extrair_paginas_pdf(caminho_pdf)
    chunks_brutos = []
    
    titulo_atual = ""
    capitulo_atual = ""
    secao_atual = ""
    artigo_em_curso = None

    for num_pagina, texto_pagina in paginas:
        # Atualiza rastreamento de Titulo, Capitulo e Secao linha a linha
        linhas = [l.strip() for l in texto_pagina.split("\n") if l.strip()]
        for idx_l, linha in enumerate(linhas):
            if REGEX_TITULO.match(linha):
                prox = linhas[idx_l + 1] if idx_l + 1 < len(linhas) and not REGEX_INICIO_ARTIGO.match(linhas[idx_l + 1]) else ""
                titulo_atual = f"{linha} - {prox}".strip(" -") if len(prox) < 80 else linha
            elif REGEX_CAPITULO.match(linha):
                prox = linhas[idx_l + 1] if idx_l + 1 < len(linhas) and not REGEX_INICIO_ARTIGO.match(linhas[idx_l + 1]) else ""
                capitulo_atual = f"{linha} - {prox}".strip(" -") if len(prox) < 80 else linha
            elif REGEX_SECAO.match(linha):
                prox = linhas[idx_l + 1] if idx_l + 1 < len(linhas) and not REGEX_INICIO_ARTIGO.match(linhas[idx_l + 1]) else ""
                secao_atual = f"{linha} - {prox}".strip(" -") if len(prox) < 80 else linha

        # Localiza todos os inicios de artigos ('Art. X') na pagina
        matches = list(REGEX_INICIO_ARTIGO.finditer(texto_pagina))

        if not matches:
            # Pagina sem inicio de novo artigo: continua artigo anterior ou indexa como bloco complementar
            if artigo_em_curso is not None and len(texto_pagina) > 30:
                artigo_em_curso["text"] += "\n" + texto_pagina
                if num_pagina not in artigo_em_curso["pages"]:
                    artigo_em_curso["pages"].append(num_pagina)
            else:
                blocos = [b.strip() for b in texto_pagina.split("\n\n") if len(b.strip()) > 40]
                if not blocos and len(texto_pagina) > 40:
                    blocos = [texto_pagina]
                for bloco in blocos:
                    chunks_brutos.append({
                        "page": num_pagina,
                        "pages": [num_pagina],
                        "artigo": "Preâmbulo / Anexos",
                        "titulo": titulo_atual,
                        "capitulo": capitulo_atual,
                        "secao": secao_atual,
                        "text": bloco
                    })
            continue

        # Texto antes do primeiro 'Art.' da pagina pertence ao artigo que vinha da pagina anterior
        prefixo_pagina = texto_pagina[:matches[0].start()].strip()
        if prefixo_pagina and len(prefixo_pagina) > 20:
            if artigo_em_curso is not None:
                artigo_em_curso["text"] += "\n" + prefixo_pagina
                if num_pagina not in artigo_em_curso["pages"]:
                    artigo_em_curso["pages"].append(num_pagina)
            else:
                chunks_brutos.append({
                    "page": num_pagina,
                    "pages": [num_pagina],
                    "artigo": "Disposições Preliminares",
                    "titulo": titulo_atual,
                    "capitulo": capitulo_atual,
                    "secao": secao_atual,
                    "text": prefixo_pagina
                })

        # Processa cada artigo encontrado na pagina
        for i, match in enumerate(matches):
            if artigo_em_curso is not None:
                chunks_brutos.append(artigo_em_curso)

            inicio_art = match.start(1)
            fim_art = matches[i + 1].start(1) if i + 1 < len(matches) else len(texto_pagina)
            corpo_artigo = texto_pagina[inicio_art:fim_art].strip()
            rotulo_art = _normalizar_rotulo_artigo(match.group(1))

            artigo_em_curso = {
                "page": num_pagina,
                "pages": [num_pagina],
                "artigo": rotulo_art,
                "titulo": titulo_atual,
                "capitulo": capitulo_atual,
                "secao": secao_atual,
                "text": corpo_artigo
            }

    if artigo_em_curso is not None:
        chunks_brutos.append(artigo_em_curso)

    # Subdivide artigos muito extensos preservando o caput do artigo em cada sub-chunk
    chunks_finais = []
    for ch in chunks_brutos:
        texto_ch = ch["text"].strip()
        hierarquia = " | ".join([h for h in [ch.get("titulo"), ch.get("capitulo"), ch.get("secao")] if h])
        cabecalho_ctx = f"[{ch['artigo']}" + (f" - {hierarquia}]\n" if hierarquia else "]\n")

        if len(texto_ch) <= max_chars_chunk:
            ch_copia = dict(ch)
            ch_copia["text"] = cabecalho_ctx + texto_ch if ch["artigo"].startswith("Art.") else texto_ch
            chunks_finais.append(ch_copia)
        else:
            # Divide nos paragrafos (§ ou Paragrafo unico) ou quebras de linha mantendo o caput
            partes = re.split(r'(?=\n\s*(?:§|Par[aá]grafo\s+[uú]nico))', texto_ch, flags=re.IGNORECASE)
            caput = partes[0].strip()
            if len(partes) == 1:
                ch_copia = dict(ch)
                ch_copia["text"] = cabecalho_ctx + texto_ch
                chunks_finais.append(ch_copia)
            else:
                bloco_acumulado = caput
                for p in partes[1:]:
                    if len(bloco_acumulado) + len(p) <= max_chars_chunk:
                        bloco_acumulado += "\n" + p.strip()
                    else:
                        ch_sub = dict(ch)
                        ch_sub["text"] = cabecalho_ctx + bloco_acumulado
                        chunks_finais.append(ch_sub)
                        bloco_acumulado = f"{caput}\n{p.strip()}"
                if bloco_acumulado:
                    ch_sub = dict(ch)
                    ch_sub["text"] = cabecalho_ctx + bloco_acumulado
                    chunks_finais.append(ch_sub)

    return chunks_finais

cdc_chunks = extrair_chunks_cdc(pdf_filename)
chunks_documento = cdc_chunks
artigos_detectados = sum(1 for c in cdc_chunks if str(c.get("artigo", "")).startswith("Art."))
motor_pdf = "PyMuPDF (fitz)" if USAR_PYMUPDF else "PyPDF2"
print(f"Extracao concluida via {motor_pdf}: {len(cdc_chunks)} chunks juridicos estruturados ({artigos_detectados} blocos de artigos mapeados).")

## 5. Motor de Busca Hibrida Real (Embeddings Semânticos + BM25 + RRF)

Combina duas estratégias complementares:
- **Busca Vetorial Densa (`gemini-embedding-001`):** Captura a intenção semântica mesmo quando o consumidor usa linguagem coloquial diferente do texto da lei.
- **Busca Lexical Probabilística (`BM25Okapi`):** Garante precisão cirúrgica para números de artigos (`Art. 49`, `Art. 18`), prazos (`30 dias`, `7 dias`) e termos técnicos.
- **Reciprocal Rank Fusion (RRF) + Limiar de Relevância:** Combina os rankings sem precisar de calibragem manual de escalas e descarta consultas fora do escopo do CDC.

In [ ]:
import hashlib
import json
import re
import unicodedata
import numpy as np
from google.genai import types
from rank_bm25 import BM25Okapi

STOP_WORDS_PT = {
    'qual', 'quais', 'como', 'onde', 'quando', 'quem', 'para', 'com', 'sem',
    'por', 'que', 'dos', 'das', 'uma', 'uns', 'sobre', 'esse', 'esta', 'meu',
    'minha', 'posso', 'tenho', 'pelo', 'pela', 'nos', 'nas', 'aos', 'isso',
    'este', 'esta', 'entre', 'depois', 'antes', 'mais', 'menos', 'muito',
    'fazer', 'feito', 'feita', 'caso', 'seja', 'sao', 'foi', 'ser', 'tem'
}

# Dicionario complementar leve para ancorar jargoes juridicos especificos no BM25
SINONIMOS_JURIDICOS_CDC = {
    "arrependi": ["desistir", "arrependimento", "fora do estabelecimento", "art 49"],
    "arrependimento": ["desistir", "sete dias", "art 49"],
    "internet": ["fora do estabelecimento", "telefone", "domicilio", "art 49"],
    "online": ["fora do estabelecimento", "telefone", "domicilio", "art 49"],
    "defeito": ["vicio", "sanado", "trinta dias", "art 18"],
    "conserto": ["vicio", "sanado", "trinta dias", "art 18"],
    "indevida": ["indebito", "dobro", "excesso", "art 42"],
    "cobranca": ["cobranca", "indebito", "dobro", "art 42"]
}

def normalizar(texto: str) -> str:
    nfkd = unicodedata.normalize('NFKD', texto)
    return "".join([c for c in nfkd if not unicodedata.combining(c)]).lower()

def tokenizar_bm25(texto: str, expandir: bool = False) -> list[str]:
    texto_norm = normalizar(texto)
    tokens = [t for t in re.findall(r'\b\w{2,}\b', texto_norm) if t not in STOP_WORDS_PT]
    if expandir:
        extras = []
        for t in tokens:
            if t in SINONIMOS_JURIDICOS_CDC:
                for sin in SINONIMOS_JURIDICOS_CDC[t]:
                    extras.extend(re.findall(r'\b\w{2,}\b', normalizar(sin)))
        tokens.extend(extras)
    return tokens

class IndiceHibridoCDC:
    def __init__(self, chunks: list[dict], client_genai, embedding_model: str = "gemini-embedding-001"):
        self.chunks = chunks
        self.client = client_genai
        self.embedding_model = embedding_model
        self.corpus_tokens = [tokenizar_bm25(c["text"]) for c in chunks]
        self.bm25 = BM25Okapi(self.corpus_tokens) if self.corpus_tokens else None
        self.embeddings = self._carregar_ou_gerar_embeddings()

    def _gerar_hash_corpus(self) -> str:
        conteudo = "|".join([f"{c['page']}:{c['text']}" for c in self.chunks])
        return hashlib.sha256(conteudo.encode('utf-8')).hexdigest()[:16]

    def _carregar_ou_gerar_embeddings(self) -> np.ndarray | None:
        if not self.chunks:
            return None
        cache_file = f"cache_embeddings_cdc_{self._gerar_hash_corpus()}.json"
        if os.path.exists(cache_file):
            try:
                with open(cache_file, "r", encoding="utf-8") as f:
                    dados = json.load(f)
                matriz = np.array(dados, dtype=np.float32)
                print(f"Embeddings carregados do cache local ({cache_file}): {matriz.shape}")
                return matriz
            except Exception:
                pass

        print(f"Gerando embeddings vetoriais para {len(self.chunks)} chunks com {self.embedding_model}...")
        vetores = []
        batch_size = 50
        try:
            for i in range(0, len(self.chunks), batch_size):
                lote = [c["text"] for c in self.chunks[i:i + batch_size]]
                resp = self.client.models.embed_content(
                    model=self.embedding_model,
                    contents=lote,
                    config=types.EmbedContentConfig(task_type="RETRIEVAL_DOCUMENT")
                )
                for emb in resp.embeddings:
                    v = np.array(emb.values, dtype=np.float32)
                    norma = np.linalg.norm(v)
                    vetores.append((v / norma if norma > 0 else v).tolist())

            with open(cache_file, "w", encoding="utf-8") as f:
                json.dump(vetores, f)
            matriz = np.array(vetores, dtype=np.float32)
            print(f"Indexacao vetorial concluida e salva em cache: {matriz.shape}")
            return matriz
        except Exception as e:
            print(f"Aviso: Falha ao gerar embeddings ({e}). Operando em modo BM25 lexical.")
            return None

    def _embed_query(self, query: str) -> np.ndarray | None:
        if self.embeddings is None:
            return None
        try:
            resp = self.client.models.embed_content(
                model=self.embedding_model,
                contents=query,
                config=types.EmbedContentConfig(task_type="RETRIEVAL_QUERY")
            )
            v = np.array(resp.embeddings[0].values, dtype=np.float32)
            norma = np.linalg.norm(v)
            return v / norma if norma > 0 else v
        except Exception:
            return None

    def buscar(self, query: str, top_k: int = 4, rrf_k: int = 60, min_cosine: float = 0.56) -> list[dict]:
        if not self.chunks:
            return []

        n_docs = len(self.chunks)
        tokens_query = tokenizar_bm25(query, expandir=True)
        bm25_scores = self.bm25.get_scores(tokens_query) if (self.bm25 and tokens_query) else np.zeros(n_docs)
        max_bm25 = float(np.max(bm25_scores)) if n_docs > 0 else 0.0

        vetor_query = self._embed_query(query)
        if vetor_query is not None and self.embeddings is not None:
            cosine_scores = np.dot(self.embeddings, vetor_query)
            max_cosine = float(np.max(cosine_scores))
        else:
            cosine_scores = np.zeros(n_docs)
            max_cosine = 0.0

        # Guardrail de escopo: se nem o BM25 nem a similaridade semantica atingirem o limiar minimo, recusa
        if max_bm25 < 1.5 and max_cosine < min_cosine:
            return []

        # Ranking BM25 (apenas documentos com score > 0 contribuem no RRF lexical)
        ordem_bm25 = np.argsort(bm25_scores)[::-1]
        rank_bm25 = {int(idx): pos + 1 for pos, idx in enumerate(ordem_bm25) if bm25_scores[idx] > 0}

        # Ranking Vetorial (apenas documentos acima de um piso minimo de similaridade)
        ordem_vec = np.argsort(cosine_scores)[::-1]
        rank_vec = {int(idx): pos + 1 for pos, idx in enumerate(ordem_vec) if cosine_scores[idx] >= (min_cosine - 0.05)}

        # Reciprocal Rank Fusion (RRF): score = peso_vec / (k + rank_vec) + peso_bm25 / (k + rank_bm25)
        candidatos = set(rank_bm25.keys()) | set(rank_vec.keys())
        resultados_rrf = []
        for idx in candidatos:
            score_rrf = 0.0
            if idx in rank_vec:
                score_rrf += 1.2 / (rrf_k + rank_vec[idx])
            if idx in rank_bm25:
                score_rrf += 1.0 / (rrf_k + rank_bm25[idx])

            chunk_enriquecido = dict(self.chunks[idx])
            chunk_enriquecido["score_rrf"] = round(score_rrf, 5)
            chunk_enriquecido["score_cosine"] = round(float(cosine_scores[idx]), 4)
            chunk_enriquecido["score_bm25"] = round(float(bm25_scores[idx]), 4)
            resultados_rrf.append((score_rrf, chunk_enriquecido))

        resultados_rrf.sort(key=lambda item: item[0], reverse=True)
        return [item[1] for item in resultados_rrf[:top_k]]

# Inicializa o indice hibrido com os chunks extraidos do PDF
indice_hibrido = IndiceHibridoCDC(cdc_chunks, client)

def buscar_chunks_cdc(query: str, chunks: list[dict] = None, top_k: int = 4) -> list[dict]:
    global indice_hibrido
    if chunks is not None and len(chunks) != len(indice_hibrido.chunks):
        indice_hibrido = IndiceHibridoCDC(chunks, client)
    return indice_hibrido.buscar(query, top_k=top_k)

# Alias de compatibilidade
buscar_paragrafos = buscar_chunks_cdc

## 6. Saida Estruturada com Pydantic e Modelo Ultra-Rapido (Gemini 3.5 Flash-Lite)

Utiliza o modelo `gemini-3.5-flash-lite` aliado a um schema **Pydantic (`RespostaJuridicaCDC`)** validado via JSON Schema:
- Garante que o modelo separe formalmente o indicador de fundamentacao (`fundamentacao_encontrada`), a orientacao ao consumidor (`resposta_orientativa`), a lista de artigos (`artigos_citados`), os numeros exatos de pagina (`paginas_pdf`) e a citacao literal (`trecho_literal`).
- Inclui validacao cruzada automatica: apenas paginas efetivamente presentes nos chunks recuperados sao mantidas, barrando qualquer alucinacao de pagina.

In [ ]:
from pydantic import BaseModel, Field
from google.genai import types

MSG_RECUSA_PADRAO = "Não encontrei essa informação no documento."

class RespostaJuridicaCDC(BaseModel):
    fundamentacao_encontrada: bool = Field(
        description="True apenas se a resposta para a duvida estiver expressamente contida nos trechos do CDC fornecidos. False caso contrario."
    )
    resposta_orientativa: str = Field(
        description="Orientacao clara, direta e fundamentada ao consumidor com base exclusiva no texto fornecido. Se fundamentacao_encontrada for False, preencha exatamente: 'Não encontrei essa informação no documento.'"
    )
    artigos_citados: list[str] = Field(
        default_factory=list,
        description="Lista dos artigos e paragrafos do CDC utilizados na resposta (ex: ['Art. 49', 'Art. 49, Parágrafo único']). Lista vazia se nao houver fundamentacao."
    )
    paginas_pdf: list[int] = Field(
        default_factory=list,
        description="Lista com os numeros inteiros das paginas do PDF de onde a informacao foi extraida (ex: [29]). Lista vazia se nao houver fundamentacao."
    )
    trecho_literal: str = Field(
        default="",
        description="Citacao literal curta (ipsis litteris) do trecho da lei que comprova a resposta. Vazio se fundamentacao_encontrada for False."
    )

def formatar_resposta_estruturada(dados: RespostaJuridicaCDC) -> str:
    if not dados.fundamentacao_encontrada or MSG_RECUSA_PADRAO.lower() in dados.resposta_orientativa.lower():
        return MSG_RECUSA_PADRAO

    pags_str = ", ".join(f"Página {p}" for p in dados.paginas_pdf) if dados.paginas_pdf else "Página não especificada"
    arts_str = ", ".join(dados.artigos_citados) if dados.artigos_citados else "Artigo aplicável do CDC"

    linhas = [
        f"{dados.resposta_orientativa.strip()}",
        f"",
        f"Fundamentação Legal: {arts_str} ({pags_str})."
    ]
    if dados.trecho_literal and dados.trecho_literal.strip():
        linhas.append(f'Trecho Literal: "{dados.trecho_literal.strip()}"')
    return "\n".join(linhas)

def perguntar_ao_assistente_estruturado(
    pergunta: str,
    chunks: list[dict],
    model_name: str = "gemini-3.5-flash-lite"
) -> RespostaJuridicaCDC:
    trechos_recuperados = buscar_chunks_cdc(pergunta, chunks, top_k=4)

    if not trechos_recuperados:
        return RespostaJuridicaCDC(
            fundamentacao_encontrada=False,
            resposta_orientativa=MSG_RECUSA_PADRAO,
            artigos_citados=[],
            paginas_pdf=[],
            trecho_literal=""
        )

    paginas_validas = set()
    blocos_contexto = []
    for c in trechos_recuperados:
        pag_principal = int(c.get('page', c.get('pagina', 0)))
        pags_lista = [int(p) for p in c.get('pages', [pag_principal]) if p]
        paginas_validas.update(pags_lista)
        rotulo_pag = f"Pagina {pag_principal}" if len(pags_lista) <= 1 else f"Pagina {pag_principal} (abrange pags. {', '.join(map(str, pags_lista))})"
        rotulo_art = c.get('artigo', '')
        cabecalho = f"--- [{rotulo_pag} | {rotulo_art}] ---" if rotulo_art else f"--- [{rotulo_pag}] ---"
        blocos_contexto.append(f"{cabecalho}\n{c['text'] if 'text' in c else c['texto']}")
    contexto = "\n\n".join(blocos_contexto)

    prompt = f"""Você é um assistente jurídico especializado exclusivamente no Código de Defesa do Consumidor (CDC).
Analise o TEXTO abaixo extraído do PDF oficial do CDC e responda à PERGUNTA preenchendo o schema JSON solicitado.

REGRAS OBRIGATÓRIAS:
1. Utilize APENAS as informações presentes no TEXTO abaixo.
2. Se a resposta não estiver contida no TEXTO abaixo (por exemplo, dúvidas sobre CLT/Direito do Trabalho, Direito Penal ou assuntos fora dos trechos), defina `fundamentacao_encontrada` como `false`, `resposta_orientativa` exatamente como "{MSG_RECUSA_PADRAO}", e deixe `artigos_citados`, `paginas_pdf` e `trecho_literal` vazios.
3. Quando `fundamentacao_encontrada` for `true`, cite em `paginas_pdf` o número exato da página indicada no cabeçalho `--- [Pagina X | Art. Y] ---` do trecho utilizado.

TEXTO:
{contexto}

PERGUNTA:
{pergunta}
"""

    dados_validados = None

    # 1. Metodo prioritario via Interactions API com JSON Schema Pydantic
    try:
        interaction = client.interactions.create(
            model=model_name,
            input=prompt,
            response_format={
                "type": "text",
                "mime_type": "application/json",
                "schema": RespostaJuridicaCDC.model_json_schema()
            }
        )
        if hasattr(interaction, 'output_text') and interaction.output_text:
            dados_validados = RespostaJuridicaCDC.model_validate_json(interaction.output_text)
    except Exception:
        pass

    # 2. Metodo alternativo via models.generate_content com response_schema Pydantic
    if dados_validados is None:
        try:
            resposta = client.models.generate_content(
                model=model_name,
                contents=prompt,
                config=types.GenerateContentConfig(
                    response_mime_type="application/json",
                    response_schema=RespostaJuridicaCDC,
                    temperature=0.0
                )
            )
            if hasattr(resposta, 'parsed') and isinstance(resposta.parsed, RespostaJuridicaCDC):
                dados_validados = resposta.parsed
            elif hasattr(resposta, 'text') and resposta.text:
                dados_validados = RespostaJuridicaCDC.model_validate_json(resposta.text)
        except Exception as e:
            return RespostaJuridicaCDC(
                fundamentacao_encontrada=False,
                resposta_orientativa=f"Erro na chamada da API: {e}",
                artigos_citados=[],
                paginas_pdf=[],
                trecho_literal=""
            )

    if dados_validados is None:
        return RespostaJuridicaCDC(
            fundamentacao_encontrada=False,
            resposta_orientativa=MSG_RECUSA_PADRAO,
            artigos_citados=[],
            paginas_pdf=[],
            trecho_literal=""
        )

    # Validacao cruzada anti-alucinacao de paginas: filtra apenas paginas reais dos chunks recuperados
    if dados_validados.fundamentacao_encontrada:
        pags_filtradas = [p for p in dados_validados.paginas_pdf if p in paginas_validas]
        if not pags_filtradas and trechos_recuperados:
            pags_filtradas = [int(trechos_recuperados[0].get('page', trechos_recuperados[0].get('pagina', 1)))]
        dados_validados.paginas_pdf = sorted(list(dict.fromkeys(pags_filtradas)))
    else:
        dados_validados.resposta_orientativa = MSG_RECUSA_PADRAO
        dados_validados.artigos_citados = []
        dados_validados.paginas_pdf = []
        dados_validados.trecho_literal = ""

    return dados_validados

def perguntar_ao_assistente(
    pergunta: str,
    chunks: list[dict],
    model_name: str = "gemini-3.5-flash-lite",
    retornar_objeto: bool = False
) -> str | RespostaJuridicaCDC:
    resultado = perguntar_ao_assistente_estruturado(pergunta, chunks, model_name=model_name)
    if retornar_objeto:
        return resultado
    return formatar_resposta_estruturada(resultado)

# Alias de compatibilidade
consultar_direitos_consumidor = perguntar_ao_assistente

## 7. Execucao dos Casos de Teste (Entregaveis)

Executa os 4 testes de forma rapida para geracao dos prints exigidos.

In [ ]:
casos_de_teste = [
    # Teste 1: Arrependimento em compras online (Art. 49 / Pagina 29)
    {
        "id": "CASO DE TESTE 1",
        "pergunta": "Qual e o prazo para o consumidor desistir de um contrato feito pela internet ou telefone?"
    },
    # Teste 2: Prazo para conserto de defeito (Art. 18 / Pagina 16)
    {
        "id": "CASO DE TESTE 2",
        "pergunta": "Qual e o prazo maximo para o fornecedor sanar o vicio ou defeito de um produto?"
    },
    # Teste 3: Cobranca indevida em dobro (Art. 42 / Pagina 26)
    {
        "id": "CASO DE TESTE 3",
        "pergunta": "O que o consumidor tem direito de receber caso seja cobrado em quantia indevida?"
    },
    # Teste 4: Fora do escopo do CDC (Recusa padronizada)
    {
        "id": "CASO DE TESTE 4",
        "pergunta": "Qual e o prazo de aviso previo para demissao sem justa causa de acordo com a CLT?"
    }
]

for caso in casos_de_teste:
    print("=" * 80)
    print(f"{caso['id']}")
    print(f"PERGUNTA: {caso['pergunta']}")
    print("-" * 80)
    resposta = perguntar_ao_assistente(caso["pergunta"], cdc_chunks if 'cdc_chunks' in locals() else chunks_documento)
    print(f"RESPOSTA DO ASSISTENTE:\n{resposta}\n")